# 🔁 Unidad 5: Pipeline de regresión con DVC

En este notebook se mostrará cómo un flujo de modelado puede evolucionar hacia un **pipeline reproducible** apoyado en DVC. La idea central no es usar DVC solo como herramienta de versionamiento de archivos, sino como mecanismo para formalizar etapas, dependencias y salidas dentro de un proyecto de Machine Learning.


## Objetivos de aprendizaje

Al finalizar este notebook, el estudiante podrá:

- identificar las etapas mínimas de un pipeline de Machine Learning;
- estructurar dependencias y salidas de un flujo reproducible;
- conectar la lógica de modelado con archivos como `dvc.yaml` y `params.yaml`;
- reconocer el papel de DVC dentro de un proyecto MLOps.


## 1. ¿Qué problema resuelve un pipeline?

En un notebook exploratorio, el flujo de trabajo depende del orden de ejecución de las celdas. En cambio, en un pipeline bien definido, cada etapa tiene:

- **entradas** claras;
- **procesos** identificables;
- **salidas** explícitas.

Esto permite repetir el flujo bajo las mismas condiciones y facilita la automatización.


## 2. Etapas del pipeline para el caso de regresión

En nuestro caso de estudio, una estructura básica podría ser:

```{mermaid}
flowchart LR
    A[Generación o carga de datos] --> B[Preprocesamiento]
    B --> C[Entrenamiento]
    C --> D[Evaluación]
```

Cada una de estas etapas podría convertirse en un script dentro de la carpeta `src/`.


## 3. Generación del dataset base

Para mantener este notebook autocontenido, reutilizaremos un generador de datos simulado. En un proyecto real, esta etapa estaría en un script independiente como `src/data/make_dataset.py`.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE: int = 42


In [2]:
def generate_session_dataset(n_samples: int, random_state: int = 42) -> pd.DataFrame:
    """Generate a mock dataset for session duration prediction."""
    rng = np.random.default_rng(random_state)

    sites = np.array(["MLA", "MLB", "MLC", "MLM"])
    device_os = np.array(["android", "ios", "web"])
    segments = np.array(["new", "active", "at_risk", "churned"])
    entry_points = np.array(["home", "search", "recommendation", "push", "email"])

    data = pd.DataFrame({
        "user_id": np.arange(1, n_samples + 1),
        "site": rng.choice(sites, size=n_samples, p=[0.35, 0.20, 0.15, 0.30]),
        "device_os": rng.choice(device_os, size=n_samples, p=[0.55, 0.25, 0.20]),
        "segment": rng.choice(segments, size=n_samples, p=[0.20, 0.45, 0.20, 0.15]),
        "entry_point": rng.choice(entry_points, size=n_samples, p=[0.25, 0.25, 0.20, 0.20, 0.10]),
        "hour_of_day": rng.integers(0, 24, size=n_samples),
        "day_of_week": rng.integers(0, 7, size=n_samples),
        "historical_avg_session_minutes": rng.uniform(2.0, 35.0, size=n_samples),
        "historical_sessions_last_7d": rng.integers(1, 25, size=n_samples),
        "days_since_last_session": rng.integers(0, 30, size=n_samples),
        "push_received_last_24h": rng.integers(0, 6, size=n_samples),
    })

    segment_effect = data["segment"].map({
        "new": -1.5,
        "active": 3.5,
        "at_risk": -2.0,
        "churned": -4.0,
    })
    entry_effect = data["entry_point"].map({
        "home": 1.2,
        "search": 2.0,
        "recommendation": 3.0,
        "push": 1.5,
        "email": 0.8,
    })
    device_effect = data["device_os"].map({
        "android": 0.5,
        "ios": 0.8,
        "web": 1.5,
    })

    hour_effect = np.where(data["hour_of_day"].between(18, 22), 2.2, 0.0)
    weekend_effect = np.where(data["day_of_week"].isin([5, 6]), 1.3, 0.0)
    noise = rng.normal(loc=0.0, scale=2.5, size=n_samples)

    data["session_minutes"] = (
        1.5
        + 0.55 * data["historical_avg_session_minutes"]
        + 0.30 * data["historical_sessions_last_7d"]
        - 0.25 * data["days_since_last_session"]
        - 0.40 * data["push_received_last_24h"]
        + segment_effect
        + entry_effect
        + device_effect
        + hour_effect
        + weekend_effect
        + noise
    ).clip(lower=0.5)

    return data


data = generate_session_dataset(n_samples=25000, random_state=RANDOM_STATE)
data.head()


,user_id,site,device_os,segment,entry_point,hour_of_day,day_of_week,historical_avg_session_minutes,historical_sessions_last_7d,days_since_last_session,push_received_last_24h,session_minutes
0,1,MLM,web,churned,email,23,0,15.777379,10,8,2,10.856501
1,2,MLB,web,active,search,22,0,20.977217,21,16,0,27.329135
2,3,MLM,web,active,email,23,2,18.077159,21,19,3,17.907926
3,4,MLC,android,new,push,22,1,21.220637,19,11,4,16.236853
4,5,MLA,android,active,push,21,0,30.180595,19,27,2,20.304020


## 4. Separación conceptual por etapas

Un pipeline reproducible no depende de que todo esté en una sola celda. Podemos pensar el flujo en tres funciones principales:

1. preparar datos;
2. entrenar modelo;
3. evaluar resultados.

En un proyecto real, estas funciones vivirían en scripts independientes.


In [3]:
def split_dataset(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Split dataset into train and test partitions."""
    target_column = "session_minutes"
    feature_columns = [
        "site",
        "device_os",
        "segment",
        "entry_point",
        "hour_of_day",
        "day_of_week",
        "historical_avg_session_minutes",
        "historical_sessions_last_7d",
        "days_since_last_session",
        "push_received_last_24h",
    ]

    X = dataframe[feature_columns].copy()
    y = dataframe[target_column].copy()

    return train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=RANDOM_STATE,
    )


def build_regression_pipeline(params: dict) -> Pipeline:
    """Build a regression pipeline from a parameter dictionary."""
    categorical_features = ["site", "device_os", "segment", "entry_point"]
    numeric_features = [
        "hour_of_day",
        "day_of_week",
        "historical_avg_session_minutes",
        "historical_sessions_last_7d",
        "days_since_last_session",
        "push_received_last_24h",
    ]

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    regressor = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor),
        ]
    )


def evaluate_regression_model(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    """Compute evaluation metrics for a regression model."""
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "r2": r2_score(y_true, y_pred),
    }


## 5. Parametrización del modelo

Una idea central en DVC es separar la lógica del pipeline de los parámetros del experimento. Para ello, en un proyecto real se usaría un archivo `params.yaml`.

Aquí simularemos esa idea con un diccionario Python.


In [4]:
params = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 4,
}

print(json.dumps(params, indent=2))


{
  "n_estimators": 200,
  "max_depth": 10,
  "min_samples_split": 4
}


## 6. Entrenamiento del pipeline

Ahora conectaremos las funciones anteriores para simular una ejecución estructurada del pipeline.


In [5]:
X_train, X_test, y_train, y_test = split_dataset(data)
pipeline = build_regression_pipeline(params)
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
metrics = evaluate_regression_model(y_test, predictions)
metrics


C:\Users\legion\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


{'mae': 2.173471257911695,
 'rmse': 2.7443297510027533,
 'r2': 0.8536388270785888}

In [6]:
metrics_df = pd.DataFrame(
    {
        "metric": list(metrics.keys()),
        "value": list(metrics.values()),
    }
)
metrics_df


,metric,value
0,mae,2.173471
1,rmse,2.744330
2,r2,0.853639


## 7. ¿Cómo se vería esto en DVC?

Hasta ahora, el flujo ya es más ordenado que en un notebook exploratorio. Pero DVC va un paso más allá: permite declarar formalmente las etapas en un archivo `dvc.yaml`.

Un ejemplo conceptual podría ser:


In [7]:
dvc_yaml_example = '''stages:
  prepare_data:
    cmd: python src/data/make_dataset.py
    deps:
      - src/data/make_dataset.py
    outs:
      - data/raw/session_data.csv

  train_model:
    cmd: python src/models/train_model.py
    deps:
      - src/models/train_model.py
      - data/raw/session_data.csv
      - params.yaml
    outs:
      - models/regression_model.pkl

  evaluate_model:
    cmd: python src/evaluation/evaluate_model.py
    deps:
      - src/evaluation/evaluate_model.py
      - models/regression_model.pkl
      - data/raw/session_data.csv
    outs:
      - outputs/metrics.json
'''

print(dvc_yaml_example)


stages:
  prepare_data:
    cmd: python src/data/make_dataset.py
    deps:
      - src/data/make_dataset.py
    outs:
      - data/raw/session_data.csv

  train_model:
    cmd: python src/models/train_model.py
    deps:
      - src/models/train_model.py
      - data/raw/session_data.csv
      - params.yaml
    outs:
      - models/regression_model.pkl

  evaluate_model:
    cmd: python src/evaluation/evaluate_model.py
    deps:
      - src/evaluation/evaluate_model.py
      - models/regression_model.pkl
      - data/raw/session_data.csv
    outs:
      - outputs/metrics.json



## 8. Interpretación del `dvc.yaml`

Observa que en este archivo cada etapa declara:

- **cmd**: qué se ejecuta;
- **deps**: de qué archivos depende;
- **outs**: qué produce.

Esto permite que DVC sepa exactamente qué debe volver a ejecutar si cambian los datos, los parámetros o el código.


## 9. Flujo conceptual completo con DVC

```{mermaid}
flowchart LR
    A[params.yaml] --> C[train_model.py]
    B[data/raw/session_data.csv] --> C[train_model.py]
    C --> D[models/regression_model.pkl]
    D --> E[evaluate_model.py]
    B --> E[evaluate_model.py]
    E --> F[outputs/metrics.json]
```

Aquí ya no hablamos solo de código, sino de una **red explícita de dependencias**.


## 10. ¿Qué gana el proyecto con esto?

Al formalizar el flujo con DVC, el proyecto gana:

- reproducibilidad;
- trazabilidad de datos y salidas;
- control explícito de dependencias;
- facilidad para ejecutar nuevamente el pipeline.

En otras palabras, dejamos de depender del orden manual de ejecución del notebook.


## 11. Limitaciones actuales del notebook

Aunque este notebook ya representa un paso importante, todavía no es un proyecto MLOps completo. Aún faltaría:

- mover cada etapa a scripts independientes;
- definir `params.yaml` como archivo real;
- versionar datos con `dvc add`;
- registrar experimentos con MLflow;
- integrar el entorno con Poetry.


## 12. Ejercicio guiado

A partir de este notebook, realiza lo siguiente:

1. Identifica qué celdas deberían convertirse en scripts separados.
2. Propón el contenido mínimo de un archivo `params.yaml`.
3. Explica qué dependencias y salidas tendría la etapa de entrenamiento.
4. Describe qué ventajas tiene ejecutar `dvc repro` frente a repetir manualmente las celdas del notebook.


## 13. Idea clave de cierre

La lección central de esta unidad es la siguiente:

> DVC no solo ayuda a versionar datos; también permite pensar el proyecto como un pipeline explícito y reproducible.

En la siguiente unidad, este flujo se conectará con el tracking de experimentos mediante MLflow.
